In [0]:
from project_tools import merge_into_table
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import DecimalType

In [0]:
jdbc_url = dbutils.secrets.get(scope="project_key", key="jdbc_url")

connection_properties = {
    "user": "neondb_owner",
    "password": dbutils.secrets.get(scope="project_key", key="db_password"),
    "driver": "org.postgresql.Driver"
}

df = ( spark.read
                .jdbc(url=jdbc_url, table="dorm_bills", properties=connection_properties)
                .withColumn("elec_meter", col("elec_meter").cast(DecimalType(10, 2)))
                .withColumn("_processing_time", current_timestamp())
                )

merge_into_table(df, ["bill_id"], "bronze_layer")


In [0]:
df = ( spark.read
                .jdbc(url=jdbc_url, table="dorm_rates", properties=connection_properties)
                .withColumn("_processing_time", current_timestamp())
                )

merge_into_table(df, ["rate_id"], "dorm_rate")